# LiteLLM connecting to Amazon Bedrock AgentCore Gateway
## Calling Amazon Bedrock AgentCore Gateway MCP Tools from LiteLLM 

## Overview

[LiteLLM](https://docs.litellm.ai/) is a proxy server and an open-source Python library that acts as a universal API gateway for accessing 100+ large language models from different providers (Bedrock, OpenAI, Anthropic, Azure, Cohere, Mistral, Gemini, Ollama, etc.). It offers production-ready features including cost tracking, guardrails, load balancing, failover, rate limiting, and monitoring for provider-agnostic infrastructure.

LiteLLM proxy also serves as a centralized [MCP gateway](https://docs.litellm.ai/docs/mcp) for MCP server management, addressing decentralized security vulnerabilities while providing unified authentication/authorization. It eliminates duplicate OAuth/RBAC configurations, prevents configuration drift, and enables multi-server aggregation. Technical advantages include LLM-agnostic support, seamless function calling integration, and protocol bridging capabilities, creating enterprise-grade operational controls for standardized MCP connections across multiple servers and clients.

Bedrock AgentCore Gateway provides customers a way to turn their existing AWS Lambda functions into fully-managed MCP servers without needing to manage infra or hosting. Gateway will provide a uniform Model Context Protocol (MCP) interface across all these tools. 

One of AWS ISV Customers, uses LiteLLM as their entry point for accessing LLMs (AWS Bedrock, OpenAI, Huggingface etc.) for their GenAI products. LiteLLM acts as a unified orchestration layer for ISVs, allowing a single API to manage multiple LLM providers, enforce governance, optimize costs, and enable tool/agent integration, all while reducing engineering overhead. This lets ISVs focus on product value without worrying about provider-specific differences, reliability, or compliance. ISVs don’t need LiteLLM when they use a single LLM provider, have no multi-tenant requirements, and don’t require advanced governance, failover, or cost-management features.

In this example, we will demonstrate how to connect LiteLLM proxy with Bedrock AgentCore Gateway using Python SDK. This will use OAuth for Inbound Authentication to connect with Gateway. This tutorial will show MCP List Tools and Call Tool.

![How does it work](images/litellm-oauth-gateway.png)

### Tutorial Details


| Information          | Details                                                   |
|:---------------------|:----------------------------------------------------------|
| Tutorial type        | Interactive                                               |
| AgentCore components | AgentCore Gateway, AgentCore Identity                     |
| MCP components       | LiteLLM                                                   |
| Gateway Target type  | AWS Lambda                                                |
| Inbound Auth IdP     | Amazon Cognito                                            |
| Outbound Auth        | AWS IAM                                                   |
| LLM model            | Anthropic Claude Sonnet 3.7, Amazon Nova Pro              |
| Tutorial components  | Creating AgentCore Gateway and Connecting with LiteLLM    |
| Tutorial vertical    | Cross-vertical                                            |
| Example complexity   | Easy                                                      |
| SDK used             | boto3, LiteLLM Python SDK                                 |

In the first part of the tutorial we will create some AmazonCore Gateway targets

### Tutorial Architecture
In this tutorial we will transform operations defined in AWS lambda function into MCP tools and host it in Bedrock AgentCore Gateway.
For demonstration purposes, we will use LiteLLM proxy to connect to AgentCore Gateway.
In our example LiteLLM will call two tools: get_order and update_order.

## Prerequisites

To execute this tutorial you will need:
* Jupyter notebook (Python kernel)
* uv
* AWS credentials
* Amazon Cognito
* LiteLLM Python SDK

## Configuring Authentication for Incoming AgentCore Gateway Requests
AgentCore Gateway provides secure connections via inbound and outbound authentication. For the inbound authentication, the AgentCore Gateway analyzes the OAuth token passed during invocation to decide allow or deny the access to a tool in the gateway. If a tool needs access to external resources, the AgentCore Gateway can use outbound authentication via API Key, IAM or OAuth Token to allow or deny the access to the external resource.



During the inbound authorization flow, an agent or the MCP client calls an MCP tool in the AgentCore Gateway adding an OAuth access token (generated from the user’s IdP). AgentCore Gateway then validates the OAuth access token and performs inbound authorization.

If the tool running in AgentCore Gateway needs to access external resources, OAuth will retrieve credentials of downstream resources using the resource credential provider for the Gateway target. AgentCore Gateway pass the authorization credentials to the caller to get access to the downstream API. 

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Set AWS credentials if not using Amazon SageMaker notebook
import os
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_REGION', 'us-east-1') # set the AWS region

In [ ]:
import os
import sys

# Get the directory of the current script
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, '../..'))

# Add to sys.path
sys.path.insert(0, utils_dir)

# Now you can import utils
import utils

In [ ]:
#### Create a sample AWS Lambda function that you want to convert into MCP tools
lambda_resp = utils.create_gateway_lambda("lambda_function_code.zip")

if lambda_resp is not None:
    if lambda_resp['exit_code'] == 0:
        print("Lambda function created with ARN: ", lambda_resp['lambda_function_arn'])
    else:
        print("Lambda function creation failed with message: ", lambda_resp['lambda_function_arn'])

In [ ]:
#### Create an IAM role for the Gateway to assume
import utils
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role['Role']['Arn'])

# Create Amazon Cognito Pool for Inbound authorization to Gateway

In [ ]:
# Creating Cognito User Pool 
import os
import boto3
import requests
import time
from botocore.exceptions import ClientError

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access"}
]
scopeString = f"{RESOURCE_SERVER_ID}/gateway:read {RESOURCE_SERVER_ID}/gateway:write"

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {user_pool_id}")

utils.get_or_create_resource_server(cognito, user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

client_id, client_secret  = utils.get_or_create_m2m_client(cognito, user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID)
print(f"Client ID: {client_id}")

# Get discovery URL  
cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration'
print(cognito_discovery_url)

# Create the Gateway with Amazon Cognito Authorizer for inbound authorization

In [ ]:
# CreateGateway with Cognito authorizer without CMK. Use the Cognito user pool created in the previous step
gateway_client = boto3.client('bedrock-agentcore-control', region_name = os.environ['AWS_DEFAULT_REGION'])
auth_config = {
    "customJWTAuthorizer": { 
        "allowedClients": [client_id],  # Client MUST match with the ClientId configured in Cognito. Example: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": cognito_discovery_url
    }
}
create_response = gateway_client.create_gateway(name='2TestGWforLambda',
    roleArn = agentcore_gateway_iam_role['Role']['Arn'], # The IAM Role must have permissions to create/list/get/delete Gateway 
    protocolType='MCP',
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration=auth_config, 
    description='AgentCore Gateway with AWS Lambda target type'
)
print(create_response)
# Retrieve the GatewayID used for GatewayTarget creation
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)
time.sleep(10)

# Create an AWS Lambda target and transform into MCP tools

In [ ]:
# Replace the AWS Lambda function ARN below
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": lambda_resp['lambda_function_arn'], # Replace this with your AWS Lambda function ARN
            "toolSchema": {
                "inlinePayload": [
                    {
                        "name": "get_order_tool",
                        "description": "tool to get the order",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    },                    
                    {
                        "name": "update_order_tool",
                        "description": "tool to update the orderId",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    }
                ]
            }
        }
    }
}

credential_config = [ 
    {
        "credentialProviderType" : "GATEWAY_IAM_ROLE"
    }
]
targetname='LambdaUsingSDK'
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description='Lambda Target using SDK',
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config)

# Calling Bedrock AgentCore Gateway from LiteLLM 

The LiteLLM proxy seamlessly integrates with AWS tools through the Bedrock AgentCore Gateway, which implements the Model Context Protocol (MCP) specification. This integration enables secure, standardized communication between LiteLLM (MCP Client)) and Bedrock AgentCore Gateway (MCP Server)

At its core, the Bedrock AgentCore Gateway serves as a protocol-compliant Gateway that exposes fundamental MCP APIs: ListTools and InvokeTools. These APIs allow any MCP-compliant client or SDK to discover and interact with available tools in a secure, standardized way. When the Strands agent needs to access AWS services, it communicates with the Gateway using these MCP-standardized endpoints.

The Gateway's implementation adheres strictly to the [MCP Authorization specification](https://modelcontextprotocol.org/specification/draft/basic/authorization), ensuring robust security and access control. This means that every tool invocation by the LiteLLM goes through authorization step, maintaining security while enabling powerful functionality.

For example, when the LiteLLM needs to access MCP tools, it first calls ListTools to discover available tools, then uses CallTool to execute specific actions. The LiteLLM proxy SDK sample is [here](https://docs.litellm.ai/docs/mcp).The Gateway handles all the necessary security validations, protocol translations, and service interactions, making the entire process seamless and secure.

This architectural approach shows that any client or SDK that implements the MCP specification can interact with AWS services through the Gateway, making it a versatile and future-proof solution for AI agent integrations.

![Strands agent calling Gateway](images/litellm-lambda-gateway.png)

# Request the access token from Amazon Cognito for inbound authorization

In [ ]:
import time
time.sleep(10)

In [ ]:
print("Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propogation completes")
token_response = utils.get_token(user_pool_id, client_id, client_secret,scopeString,REGION)
token = token_response["access_token"]
print("Token response:", token)

# LiteLLM calling MCP tools of AWS Lambda using Bedrock AgentCore Gateway

In [ ]:
from litellm.experimental_mcp_client.client import MCPClient
from mcp.types import CallToolRequestParams

In [ ]:
# Initialize LiteLLM MCP client with HTTP transport and Bearer token auth
client = MCPClient(
    server_url=gatewayURL,
    transport_type="http",
    auth_type="bearer_token",
    auth_value=token,
)

In [ ]:
#Connecting to AgentCore Gateway (MCP Server)
await client.connect()
print("\n✓ Connected successfully!")
print("\nAvailable Tools:")
print("-" * 50)

# List all available tools
tools = await client.list_tools()

if tools:
    for i, tool in enumerate(tools, 1):
        print(f"\n{i}. {tool.name}")
        print(f"   Description: {tool.description}")
        if hasattr(tool, 'inputSchema'):
            print(f"   Input Schema: {tool.inputSchema}")
else:
    print("No tools available.")

print("\n" + "-" * 50)
print(f"Total tools: {len(tools)}")


# Call the __get_order_tool with orderId argument
tool_name = targetname + "___get_order_tool"

print("\n" + "=" * 50)
print(f"Calling {tool_name} with orderId=123...")
print("=" * 50)

# Create the CallToolRequestParams object
call_params = CallToolRequestParams(
    name=tool_name,
    arguments={"orderId": "123"}
)

tool_result = await client.call_tool(call_params)

# Access CallToolResult using dot notation (Pydantic model)
if tool_result.content and len(tool_result.content) > 0:
    print(f"Tool Call result: {tool_result.content[0].text}")
else:
    print("No content in tool result")

**Issue: if you get below error while executing below cell, it indicates incompatibily between pydantic and pydantic-core versions.**

```
TypeError: model_schema() got an unexpected keyword argument 'generic_origin'
```
**How to resolve?**

You will need to make sure you have pydantic==2.7.2 and pydantic-core 2.27.2 that are both compatible. Restart the kernel once done.

# Clean up

Additional resources are also created like IAM role, IAM Policies, Credentials provider, AWS Lambda functions, Cognito user pools, s3 buckets that you might need to manually delete as part of the clean up. This depends on the example you run.

## Delete the gateway (Optional)

In [ ]:
import utils
utils.delete_gateway(gateway_client,gatewayID)